# C10-competition-craft — Practice p17 — Solution

**Type:** challenge · **Difficulty:** advanced · **Concepts:** writeup-quality, colab-markdown-solution-authoring, markdown-code-snippets, markdown-math-formulae, colab-coding-submission, cpu-and-gpu-round-boundary, train-test-split, f1-macro, knn, feature-scaling, sklearn-pipelines

**Budget:** 135 minutes

Build the capped mini-competition as a deliberate mixed-cell Colab submission. Use `../data/train.csv` and the pinned carve `test_size=150, random_state=SEED, stratify=y`. The baseline is exactly a scaled 5-NN pipeline on all 12 features. Log its validation macro-F1 before any selection.

Then perform exactly three moves: (1) sweep `k` over `[5, 7, 9, 11, 15]` on all 12 features, smallest `k` on ties; (2) try the seven `SIGNAL` features at the current `k` and keep them only if macro-F1 strictly improves; (3) repeat the same `k` sweep on the current feature set, again choosing the smallest `k` on ties. No extra candidates or re-carves.

Submit one artifact for each of the **six separately scored rows**.

| Row | Scored deliverable |
|---|---|
| 1 — writeup quality | One rendered text cell with **Approach**, **Intuition**, and **Alternatives**. Quote the selected recipe, pinned protocol, `final_val_f1`, at least one logged alternative, and an honest selection-on-validation limitation. |
| 2 — Colab authoring | Assemble an ordered mixed-cell response: headings/claims in text cells and runnable stages in code cells. Provide `cell_plan` listing every cell type and purpose. |
| 3 — fenced code | In a text cell, render a fenced `python` excerpt of the final `predict_labels` definition. It must match the executable function but does not replace it. |
| 4 — Markdown math | Render the macro-F1 formula in display math, use inline math for `$K$`, define the symbols, and explain why equal class weight fits the 2:1 task. |
| 5 — coding submission | Preserve the full modeling evidence: exact baseline, bounded choices, four-row `log_df`, `final_val_f1`, refit on all 600 rows, exact `predict_labels`, two-run equality, fresh-run stage trace, and the pinned CSV artifacts below. Submit the notebook itself as exactly `C10-p17.ipynb`; the grader opens that downloaded artifact and runs it fresh. |
| 6 — CPU/GPU boundary | Include a rendered declaration that Round 1 is CPU-only and Round 2 permits Colab L4/GPU; do not teach or use the later GPU workflow. |

**Banned (zero points): any supervised estimator other than `KNeighborsClassifier`, including reimplementations; more than the three pinned selection moves; reading or printing the held-back split; imports beyond `sklearn`, `numpy`, `pandas`, and `matplotlib`; enabling or claiming a Round 1 GPU.**

### Exact log semantics

`log_df` has exactly columns `step`, `change`, `val_f1`, `accepted` and exactly four rows ordered `baseline, iter-1, iter-2, iter-3`. The `change` strings are exactly `scaled 5-NN, all 12 features`, `k sweep [5, 7, 9, 11, 15]`, `try SIGNAL features`, and `repeat k sweep [5, 7, 9, 11, 15]`. `val_f1` records the candidate evaluated in that row, even if iter-2 is rejected. `accepted` means the row establishes or changes the current recipe: baseline is `True`; iter-1 is `True` iff its chosen `k` differs from 5; iter-2 is `True` iff the candidate strictly improves; iter-3 is `True` iff its chosen `k` differs from the current `k`. The best `k` from each sweep becomes current even when it equals the current value. `final_val_f1` is the macro-F1 of the recipe current after iter-3.

### Exact submitted artifacts

Write `p17_log.csv` with `index=False`; it must contain exactly `log_df` under the pinned four-column schema and row order. Write `p17_predictions.csv` with `index=False`, exactly columns `row_id,prediction`, and exactly 40 rows: `row_id` equals `X.iloc[150:190].index` in order and `prediction` equals `predict_labels(X.iloc[150:190])` in the same order. No additional columns or files substitute for these names.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 20260804
SIGNAL = ["honey_stores_kg", "autumn_hive_mass_kg", "varroa_mite_index",
          "forager_traffic_per_min", "brood_frames", "daily_temp_swing_c",
          "queen_age_years"]
sweep_ks = np.array([5, 7, 9, 11, 15])

df = pd.read_csv("../data/train.csv")
FEATURES = [c for c in df.columns if c != "outcome"]
X = df[FEATURES]
y = df["outcome"].to_numpy()
stage_trace = ["setup"]

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=150, random_state=SEED, stratify=y)

def fit_score(k, columns):
    model = Pipeline([("scale", StandardScaler()),
                      ("knn", KNeighborsClassifier(n_neighbors=int(k)))])
    model.fit(X_train[list(columns)], y_train)
    predictions = model.predict(X_val[list(columns)])
    return float(f1_score(y_val, predictions, average="macro"))

def choose_k(columns):
    scored = [(fit_score(k, columns), int(k)) for k in sweep_ks]
    best_score = max(score for score, _ in scored)
    best_k = min(k for score, k in scored if score == best_score)
    return best_k, best_score

changes = ["scaled 5-NN, all 12 features",
           "k sweep [5, 7, 9, 11, 15]",
           "try SIGNAL features",
           "repeat k sweep [5, 7, 9, 11, 15]"]
selected_features = list(FEATURES)
selected_k = 5
current_f1 = fit_score(selected_k, selected_features)
log_rows = [{"step": "baseline", "change": changes[0],
             "val_f1": current_f1, "accepted": True}]
stage_trace.append("baseline")

iter1_k, iter1_f1 = choose_k(selected_features)
log_rows.append({"step": "iter-1", "change": changes[1],
                 "val_f1": iter1_f1, "accepted": iter1_k != selected_k})
selected_k, current_f1 = iter1_k, iter1_f1
stage_trace.append("iter-1")

iter2_f1 = fit_score(selected_k, SIGNAL)
iter2_accepted = iter2_f1 > current_f1
log_rows.append({"step": "iter-2", "change": changes[2],
                 "val_f1": iter2_f1, "accepted": iter2_accepted})
if iter2_accepted:
    selected_features = list(SIGNAL)
    current_f1 = iter2_f1
stage_trace.append("iter-2")

pre_iter3_k = selected_k
iter3_k, iter3_f1 = choose_k(selected_features)
log_rows.append({"step": "iter-3", "change": changes[3],
                 "val_f1": iter3_f1, "accepted": iter3_k != pre_iter3_k})
selected_k, current_f1 = iter3_k, iter3_f1
stage_trace.append("iter-3")

log_df = pd.DataFrame(log_rows,
                      columns=["step", "change", "val_f1", "accepted"])
final_val_f1 = float(current_f1)

In [ ]:
final_model = Pipeline([("scale", StandardScaler()),
                        ("knn", KNeighborsClassifier(n_neighbors=int(selected_k)))])
final_model.fit(X[list(selected_features)], y)

def predict_labels(X_test):
    predictions = final_model.predict(X_test[list(selected_features)])
    return pd.Series(predictions, index=X_test.index)

stage_trace.append("refit")
probe = X.iloc[150:190]
probe_predictions = predict_labels(probe)
direct_probe_input = probe[list(selected_features)]
if not hasattr(final_model, "feature_names_in_"):
    direct_probe_input = direct_probe_input.to_numpy()
direct_probe_predictions = final_model.predict(direct_probe_input)
contract_checks = {
    "series": isinstance(probe_predictions, pd.Series),
    "length": len(probe_predictions) == len(probe),
    "index": probe_predictions.index.equals(probe.index),
    "vocab": set(probe_predictions.unique()) <= set(np.unique(y)),
}
assert all(contract_checks.values())
assert np.array_equal(probe_predictions.to_numpy(), direct_probe_predictions)
assert isinstance(final_model, Pipeline) and len(final_model.steps) == 2
final_scaler = final_model.steps[0][1]
final_knn = final_model.steps[1][1]
assert isinstance(final_scaler, StandardScaler)
assert isinstance(final_knn, KNeighborsClassifier)
assert final_knn.n_neighbors == int(selected_k)
assert final_model.n_features_in_ == len(selected_features)
assert final_scaler.n_features_in_ == len(selected_features)
assert final_knn.n_features_in_ == len(selected_features)
assert int(final_scaler.n_samples_seen_) == len(X)
assert final_knn._fit_X.shape == (len(X), len(selected_features))
contract_ok = all(contract_checks.values())

In [ ]:
def run_submission():
    rebuilt_X_train, rebuilt_X_val, rebuilt_y_train, rebuilt_y_val = train_test_split(
        X, y, test_size=150, random_state=SEED, stratify=y)

    def rebuilt_fit_score(k, columns):
        candidate = Pipeline([("scale", StandardScaler()),
                              ("knn", KNeighborsClassifier(n_neighbors=int(k)))])
        candidate.fit(rebuilt_X_train[list(columns)], rebuilt_y_train)
        predictions = candidate.predict(rebuilt_X_val[list(columns)])
        return float(f1_score(rebuilt_y_val, predictions, average="macro"))

    def rebuilt_choose_k(columns):
        scored = [(rebuilt_fit_score(k, columns), int(k)) for k in sweep_ks]
        best_score = max(score for score, _ in scored)
        best_k = min(k for score, k in scored if score == best_score)
        return best_k, best_score

    rebuilt_features = list(FEATURES)
    rebuilt_k = 5
    rebuilt_f1 = rebuilt_fit_score(rebuilt_k, rebuilt_features)
    rebuilt_rows = [{"step": "baseline", "change": changes[0],
                     "val_f1": rebuilt_f1, "accepted": True}]

    rebuilt_iter1_k, rebuilt_iter1_f1 = rebuilt_choose_k(rebuilt_features)
    rebuilt_rows.append({"step": "iter-1", "change": changes[1],
                         "val_f1": rebuilt_iter1_f1,
                         "accepted": rebuilt_iter1_k != rebuilt_k})
    rebuilt_k, rebuilt_f1 = rebuilt_iter1_k, rebuilt_iter1_f1

    rebuilt_iter2_f1 = rebuilt_fit_score(rebuilt_k, SIGNAL)
    rebuilt_iter2_accepted = rebuilt_iter2_f1 > rebuilt_f1
    rebuilt_rows.append({"step": "iter-2", "change": changes[2],
                         "val_f1": rebuilt_iter2_f1,
                         "accepted": rebuilt_iter2_accepted})
    if rebuilt_iter2_accepted:
        rebuilt_features = list(SIGNAL)
        rebuilt_f1 = rebuilt_iter2_f1

    rebuilt_pre_iter3_k = rebuilt_k
    rebuilt_iter3_k, rebuilt_iter3_f1 = rebuilt_choose_k(rebuilt_features)
    rebuilt_rows.append({"step": "iter-3", "change": changes[3],
                         "val_f1": rebuilt_iter3_f1,
                         "accepted": rebuilt_iter3_k != rebuilt_pre_iter3_k})
    rebuilt_k, rebuilt_f1 = rebuilt_iter3_k, rebuilt_iter3_f1

    rebuilt_model = Pipeline([("scale", StandardScaler()),
                              ("knn", KNeighborsClassifier(n_neighbors=rebuilt_k))])
    rebuilt_model.fit(X[rebuilt_features], y)
    rebuilt_probe = X.iloc[150:190]
    rebuilt_predictions = pd.Series(
        rebuilt_model.predict(rebuilt_probe[rebuilt_features]),
        index=rebuilt_probe.index)
    rebuilt_log = pd.DataFrame(
        rebuilt_rows, columns=["step", "change", "val_f1", "accepted"])
    return (rebuilt_log, float(rebuilt_f1), list(rebuilt_features),
            int(rebuilt_k), rebuilt_predictions)
def _grader_fit_score(k, columns, X_train, y_train, X_valid, y_valid):
    model = Pipeline([("scale", StandardScaler()),
                      ("knn", KNeighborsClassifier(n_neighbors=int(k)))])
    model.fit(X_train[list(columns)], y_train)
    predictions = model.predict(X_valid[list(columns)])
    return float(f1_score(y_valid, predictions, average="macro"))

_grader_X_tr, _grader_X_val, _grader_y_tr, _grader_y_val = train_test_split(
    X, y, test_size=150, random_state=SEED, stratify=y)

def _grader_sweep(columns):
    scores = [(_grader_fit_score(k, columns, _grader_X_tr, _grader_y_tr,
                                 _grader_X_val, _grader_y_val), int(k))
              for k in sweep_ks]
    best_score = max(score for score, _ in scores)
    best_k = min(k for score, k in scores if score == best_score)
    return best_k, best_score

_grader_all_features = list(FEATURES)
_grader_baseline_f1 = _grader_fit_score(
    5, _grader_all_features, _grader_X_tr, _grader_y_tr, _grader_X_val, _grader_y_val)
_grader_iter1_k, _grader_iter1_f1 = _grader_sweep(_grader_all_features)
_grader_current_k = _grader_iter1_k
_grader_current_features = _grader_all_features
_grader_current_f1 = _grader_iter1_f1
_grader_iter2_f1 = _grader_fit_score(
    _grader_current_k, SIGNAL, _grader_X_tr, _grader_y_tr, _grader_X_val, _grader_y_val)
_grader_iter2_accepted = _grader_iter2_f1 > _grader_current_f1
if _grader_iter2_accepted:
    _grader_current_features = list(SIGNAL)
    _grader_current_f1 = _grader_iter2_f1
_grader_pre_iter3_k = _grader_current_k
_grader_iter3_k, _grader_iter3_f1 = _grader_sweep(_grader_current_features)
_grader_iter3_accepted = _grader_iter3_k != _grader_pre_iter3_k
_grader_current_k = _grader_iter3_k
_grader_current_f1 = _grader_iter3_f1
_grader_steps = ["baseline", "iter-1", "iter-2", "iter-3"]
_grader_changes = ["scaled 5-NN, all 12 features",
                   "k sweep [5, 7, 9, 11, 15]",
                   "try SIGNAL features",
                   "repeat k sweep [5, 7, 9, 11, 15]"]
_grader_scores = [_grader_baseline_f1, _grader_iter1_f1,
                  _grader_iter2_f1, _grader_iter3_f1]
_grader_accepted = [True, _grader_iter1_k != 5,
                    bool(_grader_iter2_accepted), bool(_grader_iter3_accepted)]
_grader_log = pd.DataFrame({"step": _grader_steps, "change": _grader_changes,
                            "val_f1": _grader_scores, "accepted": _grader_accepted})

assert isinstance(log_df, pd.DataFrame)
assert list(log_df.columns) == ["step", "change", "val_f1", "accepted"]
assert log_df.index.tolist() == [0, 1, 2, 3]
assert log_df["step"].tolist() == _grader_steps
assert log_df["change"].tolist() == _grader_changes
assert pd.api.types.is_float_dtype(log_df["val_f1"].dtype)
assert np.allclose(log_df["val_f1"], _grader_scores, atol=1e-12, rtol=0)
assert pd.api.types.is_bool_dtype(log_df["accepted"].dtype)
assert log_df["accepted"].tolist() == _grader_accepted
assert list(selected_features) == _grader_current_features
assert int(selected_k) == _grader_current_k
assert isinstance(final_val_f1, float)
assert np.allclose(final_val_f1, _grader_current_f1, atol=1e-12, rtol=0)

run_a = run_submission()
run_b = run_submission()
for rebuilt in (run_a, run_b):
    assert isinstance(rebuilt, tuple) and len(rebuilt) == 5
    assert rebuilt[0].equals(_grader_log)
    assert rebuilt[1] == _grader_current_f1
    assert list(rebuilt[2]) == _grader_current_features
    assert int(rebuilt[3]) == _grader_current_k
    assert rebuilt[4].equals(probe_predictions)
assert run_a[0].equals(run_b[0]) and run_a[0].equals(log_df)
assert run_a[1] == run_b[1] == final_val_f1
assert list(run_a[2]) == list(run_b[2]) == list(selected_features)
assert run_a[3] == run_b[3] == selected_k
assert run_a[4].equals(run_b[4]) and run_a[4].equals(probe_predictions)
cell_plan = [
    ("text", "rendered Approach, Intuition, Alternatives, and limitation"),
    ("code", "imports, constants, data load, and pinned carve"),
    ("code", "baseline and exactly three bounded selection moves"),
    ("code", "full-data refit and executable predict_labels contract"),
    ("code", "independent rebuild and two-run equality checks"),
    ("text", "matching fenced predict_labels excerpt"),
    ("text", "rendered macro-F1 formula and interpretation"),
    ("code", "write and verify exact log and prediction CSV artifacts"),
    ("text", "Round 1 CPU and Round 2 L4/GPU declaration"),
]

In [ ]:
log_df.to_csv("p17_log.csv", index=False)
pd.DataFrame({"row_id": probe.index,
              "prediction": probe_predictions.to_numpy()}).to_csv(
    "p17_predictions.csv", index=False)
stage_trace.append("package")

saved_log = pd.read_csv("p17_log.csv")
saved_predictions = pd.read_csv("p17_predictions.csv")
assert list(saved_log.columns) == ["step", "change", "val_f1", "accepted"]
assert len(saved_log) == 4
assert saved_log["step"].tolist() == _grader_steps
assert saved_log["change"].tolist() == _grader_changes
assert np.allclose(saved_log["val_f1"], _grader_scores, atol=1e-12, rtol=0)
assert saved_log["accepted"].tolist() == _grader_accepted
assert list(saved_predictions.columns) == ["row_id", "prediction"]
assert len(saved_predictions) == 40
assert saved_predictions["row_id"].tolist() == probe.index.tolist()
assert saved_predictions["prediction"].tolist() == direct_probe_predictions.tolist()
assert stage_trace == ["setup", "baseline", "iter-1", "iter-2", "iter-3", "refit", "package"]

## Row 1 — Mini-competition writeup

### Approach

*Write your reproducible recipe, protocol, refit, and score here.*

### Intuition

*Ground the model and metric choices in this data and task.*

### Alternatives

*Quote logged alternatives and state a limitation.*

## Row 3 — Fenced `predict_labels` excerpt

*Insert the rendered fenced Python excerpt here.*

## Row 4 — Rendered metric

*Insert and interpret the display formula here.*

## Row 6 — Round declaration

*State the exact CPU/L4-GPU boundary here.*

## Row 1 — Mini-competition writeup

### Approach

Using the single pinned stratified carve (`test_size=150`, `random_state=20260804`), the exact scaled 5-NN baseline on all 12 features scored macro-F1 `0.7665823769694612`. The first bounded sweep selected `k=11`, the strict-improvement move kept the seven `SIGNAL` features, and the repeated sweep retained `k=11`. The selected recipe is therefore scaled 11-NN on `SIGNAL`, with `final_val_f1 = 0.8198760747394207`, followed by a refit on all 600 labeled rows.

### Intuition

Scaling keeps large-unit features from dominating neighbor distances. The bounded `k` sweep balances noisy local votes against excessive smoothing, while macro-F1 gives each class equal influence in this 2:1 task.

### Alternatives

A logged alternative was scaled 11-NN on all 12 features at `0.8103481812876873`; restricting the same model to `SIGNAL` improved it to `0.8198760747394207`. Because all three choices reuse the same validation carve, the selected score may be optimistic and is not an untouched-test estimate.

## Row 3 — Fenced `predict_labels` excerpt

```python
def predict_labels(X_test):
    predictions = final_model.predict(X_test[list(selected_features)])
    return pd.Series(predictions, index=X_test.index)
```

## Row 4 — Rendered metric

For class $k$, let $F_{1,k}$ be its precision–recall harmonic mean. With $K$ classes,

$$
F_{1,\mathrm{macro}} = \frac{1}{K} \sum_{k=1}^{K} F_{1,k}.
$$

Here $K=2$. Equal class weight prevents the larger class from dominating the metric merely because the task has a 2:1 class balance.

## Row 6 — Round declaration

**Round 1 is CPU-only; Round 2 permits Colab L4/GPU.**

### Answer check

The immutable grader independently recomputes the frozen carve, baseline, both bounded sweeps, strict feature decision, exact four-row log, chosen features and `k`, and `final_val_f1`. It also verifies two independent rebuilds, direct `final_model` prediction equality, full-data pipeline fit state, exact CSV contents, and the seven-stage trace; all approximate comparisons use `atol=1e-12` and `rtol=0`.